In [8]:
import kagglehub
import pandas as pd
import os
# import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

## Load Data

In [18]:
path = kagglehub.dataset_download("mirichoi0218/insurance")
print("Path to dataset files:", path)
files = os.listdir(path)
print("Files found in directory:", files)

Path to dataset files: /Users/johannzaroli/.cache/kagglehub/datasets/mirichoi0218/insurance/versions/1
Files found in directory: ['insurance.csv']


In [26]:
df = pd.read_csv(os.path.join(path, "insurance.csv"))
print(df.head())

# clean df, remove NaN values
df_cleaned = df.dropna()
len(df_cleaned)

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [27]:
# encode categorical columns
encoded_df = pd.get_dummies(df_cleaned, columns=['sex', 'smoker', 'region'], dtype=int)
encoded_df.head()

,age,bmi,children,charges,sex_female,sex_male,smoker_no,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,1,0,0,1,0,0,0,1
1,18,33.770,1,1725.55230,0,1,1,0,0,0,1,0
2,28,33.000,3,4449.46200,0,1,1,0,0,0,1,0
3,33,22.705,0,21984.47061,0,1,1,0,0,1,0,0
4,32,28.880,0,3866.85520,0,1,1,0,0,1,0,0


## Feature engineering

In [29]:
encoded_df["age_bmi_smoker_risk"] =  encoded_df["age"] * encoded_df["bmi"] * encoded_df["smoker_yes"].map({0: 1.0, 1: 2.5}) 
encoded_df["bmi_x_age"] = encoded_df["bmi"] * encoded_df["age"]
encoded_df.head()

,age,bmi,children,charges,sex_female,sex_male,smoker_no,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest,age_bmi_smoker_risk,bmi_x_age
0,19,27.900,0,16884.92400,1,0,0,1,0,0,0,1,1325.250,530.100
1,18,33.770,1,1725.55230,0,1,1,0,0,0,1,0,607.860,607.860
2,28,33.000,3,4449.46200,0,1,1,0,0,0,1,0,924.000,924.000
3,33,22.705,0,21984.47061,0,1,1,0,0,1,0,0,749.265,749.265
4,32,28.880,0,3866.85520,0,1,1,0,0,1,0,0,924.160,924.160


#### Observations:
Initially, I was going to create columns for bmi_x_smoker, age_x_smoker. However, these were essentially multiplying the smoker status (0 or 1) by the other feature and creating little to no signal for the model.   
  
After careful consideration, I created a unified risk score that is age_bmi_smoker_risk score where being a smoker scales the multiplied bmi*age values by another 2.5x, modeling the compounding effects of being a smoker, being of a certain age and having a higher bmi.

I also created a bmi_x_age feature to model the compounding effects of both. This may prove to be mostly redundant with the age_bmi_smoker_risk score.

Note: The transformed features are for the polynomial model, not the random forest.

In [25]:
# X_train, X_test = train_test_split(sc_top5df, test_size=0.2, random_state=42)

NameError: name 'train_test_split' is not defined